In [ ]:
import numpy as np
import pandas as pd
import h5py
import plotly.graph_objects as go
from scipy.signal import hilbert
from scipy.ndimage import uniform_filter1d

# ------------------ 1) USER SETTINGS ------------------
PROTOCOL_FILE = r'C:/Users/megab/Downloads/All EMG recordings/indicators-24Jun2025.npy'
EMG_FILE = r'C:/Users/megab/Downloads/All EMG recordings/used/protocol2rishabtrial3.h5'
DEVICE = "98:D3:C1:FE:04:75"
FS = 1000  # Sampling rate (Hz)
CHANNEL = 'a1'  # Choose from a1, a2, a3, a4
LEFT_PAD = 0  # Seconds of 'empty' before indicators
COLOR_MAPPING = dict(
    fist='rgba(0, 0, 0, 0.8)',  # black as baseline/rest
    palm='rgba(0,255,0,0.8)',  # green
    thumb='rgba(128,192,0,0.8)',  # yellow-green
    index='rgba(192,128,0,0.8)',  # orange-yellow
    middle='rgba(255,64,0,0.8)',  # orange-red
    pinky_ring='rgba(255,0,0,0.8)',  # red
    empty='rgba(220,220,220,0.2)'  # light gray for padding
)


# ------------------ 2) EMG I/O ------------------
def reademg(filename: str, device: str = DEVICE, fs: int = FS) -> pd.DataFrame:
    with h5py.File(filename, 'r') as f:
        raw = f[device]['raw']
        data = {f'a{i+1}': raw[f'channel_{i+1}'][:, 0] for i in range(4)}
    df = pd.DataFrame(data)
    df['ts'] = np.arange(len(df)) / fs
    return df

# ------------------ 3) BUILD TIMELINE ------------------
def build_timeline(indicator_file: str, df_emg: pd.DataFrame, left_pad: float, fs: int = FS) -> list:
    gesture_map = {
        0: 'fist',
        1: 'palm',
        2: 'thumb',
        3: 'index',
        4: 'middle',
        5: 'pinky_ring'
    }

    indicators = np.load(indicator_file)

    # Replace -1s with previous valid gesture
    for i in range(len(indicators)):
        if indicators[i] == -1:
            j = i - 1
            while j >= 0 and indicators[j] == -1:
                j -= 1
            indicators[i] = indicators[j] if j >= 0 else 0

    # Estimate average durations of gestures
    gesture_durations = {}
    current_label = indicators[0]
    start_idx = 0

    for i in range(1, len(indicators)):
        if indicators[i] != current_label:
            dur = i - start_idx
            gesture_durations.setdefault(current_label, []).append(dur)
            current_label = indicators[i]
            start_idx = i
    # Final segment
    gesture_durations.setdefault(current_label, []).append(len(indicators) - start_idx)

    # Average durations per gesture
    avg_durations = {label: int(np.mean(durs)) for label, durs in gesture_durations.items()}

    # Extend indicators for missing 6 gestures: [fist, index, fist, middle, fist, pinky_ring]
    expected_sequence = [0, 3, 0, 4, 0, 5]
    extension = []
    for gesture in expected_sequence:
        duration = avg_durations.get(gesture, int(fs * 1.0))  # fallback 1s
        extension.extend([gesture] * duration)

    indicators = np.concatenate([indicators, extension])

    indicators_duration = len(indicators) / fs
    emg_duration = df_emg['ts'].iloc[-1]
    right_pad = emg_duration - left_pad - indicators_duration

    # if right_pad < 0:
    #     raise ValueError(f"Indicators + left_pad ({left_pad}s) exceed EMG duration ({emg_duration:.2f}s)")

    timeline = []

    # Add left padding
    if left_pad > 0:
        timeline.append({
            'gesture': 'empty',
            'start': 0.0,
            'end': left_pad
        })

    # Build gesture segments
    current_label = indicators[0]
    start_idx = 0
    for i in range(1, len(indicators)):
        if indicators[i] != current_label:
            end_idx = i
            label = gesture_map.get(current_label, 'unknown')
            timeline.append({
                'gesture': label,
                'start': left_pad + start_idx / fs,
                'end': left_pad + end_idx / fs
            })
            current_label = indicators[i]
            start_idx = i

    # Final gesture segment
    end_idx = len(indicators)
    label = gesture_map.get(current_label, 'unknown')
    timeline.append({
        'gesture': label,
        'start': left_pad + start_idx / fs,
        'end': left_pad + end_idx / fs
    })

    # Add right padding
    if right_pad > 0:
        timeline.append({
            'gesture': 'empty',
            'start': left_pad + indicators_duration,
            'end': emg_duration
        })

    print(f"Built timeline with {len(timeline)} intervals.")
    print(f"Left pad: {left_pad:.2f}s | Indicator duration: {indicators_duration:.2f}s "
          f"| Right pad: {right_pad:.2f}s | EMG duration: {emg_duration:.2f}s")

    return timeline

# ------------------ 4) PLOTTING FUNCTION ------------------
def plot_emg_power_with_timeline(df_emg: pd.DataFrame, timeline: list, channel: str = CHANNEL):
    ts = df_emg['ts'].values
    if channel not in df_emg.columns:
        raise ValueError(f"Channel '{channel}' not found in EMG data")

    signal = df_emg[channel].values
    signal = (signal - np.mean(signal)) / np.std(signal)

    analytic = hilbert(signal)
    power = np.log(np.abs(analytic)**2)

    window_size = int(0.5 * FS)
    smoothed_power = uniform_filter1d(power, size=window_size)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts, y=smoothed_power, mode='lines', name=f"Power ({channel.upper()})"))

    for entry in timeline:
        gesture = entry['gesture']
        color = COLOR_MAPPING.get(gesture, 'rgba(200,200,200,0.2)')
        fig.add_vrect(x0=entry['start'], x1=entry['end'], fillcolor=color, line_width=0, layer='below')

    fig.update_layout(
        title=f"EMG Power of Channel {channel.upper()} with Gesture Windows",
        xaxis_title="Time (s)",
        yaxis_title="Power (0.5s window avg)",
        xaxis=dict(rangeslider=dict(visible=True))
    )
    fig.show()

# ------------------ 5) MAIN ------------------
if __name__ == '__main__':
    df_emg = reademg(EMG_FILE, fs=FS)
    timeline = build_timeline(PROTOCOL_FILE, df_emg, left_pad=1.59, fs=FS)
    plot_emg_power_with_timeline(df_emg, timeline)


In [ ]:
import json

def export_timeline_to_json(timeline, out_path):
    """
    Given `timeline` as returned by build_timeline(), writes a JSON file
    with each dict formatted like:
      {"gesture": "...", "start": 12.34, "end": 23.45}
    """
    # Ensure floats are nicely formatted
    def fmt(x):
        return float(f"{x:.2f}")

    clean_tl = []
    for entry in timeline:
        clean_tl.append({
            "gesture": entry["gesture"],
            "start": fmt(entry["start"]),
            "end": fmt(entry["end"])
        })

    with open(out_path, "w") as f:
        json.dump(clean_tl, f, indent=2)

# ------------------ USAGE ------------------
if __name__ == "__main__":
    # after you’ve built your df_emg and timeline:
    # df_emg = reademg(...)
    # timeline = build_timeline(...)
    export_timeline_to_json(timeline, "shubProtocol2trial3timeline.json")
    print("Wrote timeline.json with", len(timeline), "entries.")


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import h5py
import plotly.graph_objects as go
from scipy.signal import hilbert
from scipy.ndimage import uniform_filter1d

# ------------------ 1) USER SETTINGS ------------------
PROTOCOL_FILE = r'C:/Users/megab/Downloads/All EMG recordings/indicators-24Jun2025.npy'

# Only HDF5 inputs now, each with its left‑pad (in seconds):
EMG_INPUTS = [
    (r"C:/Users/megab/Downloads/All EMG recordings/used/ShubProtocol2trial1.h5",9),
    (r"C:/Users/megab/Downloads/All EMG recordings/used/ShubProtocol2trial2.h5",9),
    (r"C:/Users/megab/Downloads/All EMG recordings/used/ShubProtocol2trial3.h5",4)
]

FS = 1000  # Sampling rate (Hz)
CHANNEL = 'a1'  # Which channel to plot

COLOR_MAPPING = dict(
    fist='rgba(0, 0, 0, 0.8)',
    palm='rgba(0,255,0,0.8)',
    thumb='rgba(128,192,0,0.8)',
    index='rgba(192,128,0,0.8)',
    middle='rgba(255,64,0,0.8)',
    pinky_ring='rgba(255,0,0,0.8)',
    empty='rgba(220,220,220,0.2)'
)

# ------------------ 2) EMG I/O ------------------
def read_h5_emg(path: str, fs: int = FS) -> pd.DataFrame:
    with h5py.File(path, 'r') as f:
        dev = next(iter(f.keys()))
        raw = f[dev]['raw']
        data = {f'a{i+1}': raw[f'channel_{i+1}'][:, 0] for i in range(4)}
    df = pd.DataFrame(data)
    df['ts'] = np.arange(len(df)) / fs
    return df

# ------------------ 3) BUILD TIMELINE ------------------
def build_timeline(indicator_file: str,
                   df_emg: pd.DataFrame,
                   left_pad: float,
                   fs: int = FS) -> list:
    gesture_map = {0:'fist',1:'palm',2:'thumb',3:'index',4:'middle',5:'pinky_ring'}
    inds = np.load(indicator_file)
    # fill -1 with last valid
    for i in range(len(inds)):
        if inds[i]==-1:
            j=i-1
            while j>=0 and inds[j]==-1: j-=1
            inds[i] = inds[j] if j>=0 else 0

    # compute average durations
    durs = {}; cur=inds[0]; start=0
    for i in range(1,len(inds)):
        if inds[i]!=cur:
            durs.setdefault(cur,[]).append(i-start)
            cur, start = inds[i], i
    durs.setdefault(cur,[]).append(len(inds)-start)
    avg = {g:int(np.mean(ds)) for g,ds in durs.items()}

    # extend missing gestures in sequence
    seq=[0,3,0,4,0,5]
    ext=[]
    for g in seq:
        ext += [g]*avg.get(g, int(fs*1.0))
    inds = np.concatenate([inds, ext])

    total_sec = len(inds)/fs
    emg_dur   = df_emg['ts'].iloc[-1]
    right_pad = emg_dur - left_pad - total_sec

    timeline=[]
    if left_pad>0:
        timeline.append({'gesture':'empty','start':0.0,'end':left_pad})

    cur=inds[0]; start=0
    for i in range(1,len(inds)):
        if inds[i]!=cur:
            timeline.append({
                'gesture': gesture_map[cur],
                'start':   left_pad + start/fs,
                'end':     left_pad + i/fs
            })
            cur, start = inds[i], i

    timeline.append({
        'gesture': gesture_map[cur],
        'start':   left_pad + start/fs,
        'end':     left_pad + len(inds)/fs
    })
    if right_pad>0:
        timeline.append({
            'gesture':'empty',
            'start': left_pad + total_sec,
            'end':   emg_dur
        })

    return timeline

# ------------------ 4) PLOT ------------------
def plot_emg_with_timeline(df, timeline, channel=CHANNEL):
    ts    = df['ts'].values
    sig   = df[channel].values
    sig   = (sig - sig.mean())/sig.std()
    analytic = hilbert(sig)
    power    = np.log(np.abs(analytic)**2)
    smooth   = uniform_filter1d(power, size=int(0.5*FS))

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts, y=smooth,
                             mode='lines',
                             name=channel.upper()))
    for seg in timeline:
        fig.add_vrect(x0=seg['start'], x1=seg['end'],
                      fillcolor=COLOR_MAPPING[seg['gesture']],
                      line_width=0, layer='below')
    fig.update_layout(title=f"EMG Power: {channel.upper()}",
                      xaxis_title="Time (s)",
                      yaxis_title="Log‐power (0.5s avg)",
                      xaxis=dict(rangeslider=dict(visible=True)))
    fig.show()

# ------------------ 5) MAIN ------------------
if __name__ == '__main__':
    for path, left_pad in EMG_INPUTS:
        print(f"\n--- {os.path.basename(path)} (left_pad={left_pad}s) ---")
        df_emg = read_h5_emg(path)
        tl     = build_timeline(PROTOCOL_FILE, df_emg, left_pad)
        plot_emg_with_timeline(df_emg, tl, channel=CHANNEL)
